<a href="https://colab.research.google.com/github/ianvogel-glitch/Datascience_mkt/blob/main/FeedBack_laboratorio_marketing_digital_python_comentado_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratório de Ciência de Dados aplicado ao Marketing Digital

## Notebook Python | Profa. Kadidja Valéria (análise do Laboratório - Avaliação 2)

Neste notebook, analisaremos uma base fictícia de campanhas digitais. O roteiro foi organizado de forma progressiva: carregamento, diagnóstico, tratamento, cálculo de métricas, agrupamento, visualização, interpretação e exportação.

### Métricas calculadas

- **CTR:** cliques divididos por impressões.
- **Taxa de conversão:** conversões divididas por cliques.
- **CPC:** investimento dividido por cliques.
- **CPA:** investimento dividido por conversões.
- **ROAS:** receita dividida por investimento.

> Execute as células na ordem. Leia os comentários iniciados por `#`, pois eles explicam cada comando.


## 1 Preparação do ambiente

Importaremos as bibliotecas utilizadas na análise. O Pandas manipula tabelas, o NumPy auxilia nos cálculos e Matplotlib e Seaborn produzem as visualizações.


In [ ]:
# Importa o Pandas e cria o apelido pd.
import pandas as pd

# Importa o NumPy e cria o apelido np.
import numpy as np

# Importa a biblioteca básica de gráficos.
import matplotlib.pyplot as plt

# Importa o Seaborn para gráficos estatísticos com melhor acabamento visual.
import seaborn as sns

# Importa display para apresentar tabelas formatadas no notebook.
# O bloco alternativo permite testar o código também fora do Colab.
try:
    from IPython.display import display
except ImportError:
    display = print

# Define um estilo visual comum para todos os gráficos.
sns.set_theme(style="whitegrid", palette="deep")

# Define um tamanho padrão para os gráficos.
plt.rcParams["figure.figsize"] = (9, 5)

print("Bibliotecas importadas com sucesso.")


## 2 Envio do arquivo CSV

Ao executar esta célula no Google Colab, será exibido o botão **Escolher arquivos**. Se o arquivo já estiver na pasta da sessão, o notebook continuará sem solicitar novo envio.


In [ ]:
# Importa funções para trabalhar com caminhos e arquivos.
from pathlib import Path

# Define o nome esperado do arquivo.
nome_arquivo = "campanhas_marketing_digital.csv"

# Verifica se o CSV já existe na pasta atual.
if not Path(nome_arquivo).exists():
    try:
        # Este bloco funciona no Google Colab.
        from google.colab import files

        print(f"Selecione o arquivo: {nome_arquivo}")
        arquivos_enviados = files.upload()

        # Confere se o arquivo correto foi enviado.
        if nome_arquivo not in arquivos_enviados:
            raise FileNotFoundError(
                f"O arquivo enviado não possui o nome esperado: {nome_arquivo}"
            )
    except ImportError:
        # Esta mensagem aparece quando o notebook é executado fora do Colab.
        raise FileNotFoundError(
            f"Coloque o arquivo {nome_arquivo} na mesma pasta do notebook."
        )
else:
    print(f"O arquivo {nome_arquivo} já está disponível na sessão.")


## 3 Leitura da base de dados

O comando `read_csv()` lê o arquivo e cria um DataFrame. Um DataFrame é uma tabela formada por linhas e colunas.


In [ ]:
# Lê o arquivo CSV e armazena a tabela na variável df.
# O parâmetro encoding aceita arquivos salvos com ou sem a marca UTF-8 do Excel.
df = pd.read_csv(nome_arquivo, encoding="utf-8-sig")

# Mostra as cinco primeiras linhas para confirmar a leitura.
display(df.head())

print("Arquivo carregado com sucesso.")


## 4 Diagnóstico inicial

Antes de calcular métricas, devemos compreender a estrutura e a qualidade dos dados.


In [ ]:
# shape retorna uma tupla: quantidade de linhas e quantidade de colunas.
print("Dimensão da base:", df.shape)

# columns apresenta o nome de todas as variáveis.
print("\nColunas disponíveis:")
print(df.columns.tolist())

# info apresenta tipos, quantidade de valores preenchidos e uso de memória.
print("\nEstrutura da base:")
df.info()


In [ ]:
# Conta valores ausentes em cada coluna.
print("Valores ausentes por coluna:")
display(df.isna().sum().to_frame("quantidade_ausente"))

# Conta linhas completamente duplicadas.
print("Quantidade de linhas duplicadas:", df.duplicated().sum())

# Exibe as categorias originais da coluna canal.
# Observe que uma ocorrência de Instagram possui um espaço no final.
print("Canais antes da limpeza:")
print(df["canal"].unique())


In [ ]:
# describe calcula estatísticas descritivas das colunas numéricas.
# T significa transpor: as variáveis aparecem nas linhas.
display(df.describe().T)


## 5 Tratamento dos dados

Criaremos uma cópia para preservar a base original. Em seguida, removeremos espaços extras e converteremos a data. O lead ausente permanecerá como `NaN`, pois ausência de informação não significa zero.


In [ ]:
# Cria uma cópia independente da base original.
dados = df.copy()

# Remove espaços antes e depois dos nomes dos canais.
dados["canal"] = dados["canal"].str.strip()

# Converte a coluna data, inicialmente textual, para o tipo datetime.
dados["data"] = pd.to_datetime(dados["data"], errors="coerce")

# Verifica se alguma data não pôde ser convertida.
print("Datas inválidas:", dados["data"].isna().sum())

# Confere os canais depois da limpeza.
print("Canais depois da limpeza:")
print(dados["canal"].unique())

# Confere os tipos das colunas.
print("\nTipos depois do tratamento:")
print(dados.dtypes)


## 6 Cálculo das métricas por registro

A função abaixo evita divisão por zero. Quando o denominador é zero, o resultado será `NaN`, evitando valores infinitos ou enganosos.


In [ ]:
def divisao_segura(numerador, denominador):
    # Divide duas séries e substitui denominadores iguais a zero por NaN.

    # replace troca zero por NaN somente no denominador.
    denominador_seguro = denominador.replace(0, np.nan)

    # div realiza a divisão elemento a elemento.
    return numerador.div(denominador_seguro)


# CTR: porcentagem de impressões que geraram clique.
dados["ctr_pct"] = divisao_segura(
    dados["cliques"], dados["impressoes"]
) * 100

# Taxa de conversão: porcentagem de cliques que geraram conversão.
dados["taxa_conversao_pct"] = divisao_segura(
    dados["conversoes"], dados["cliques"]
) * 100

# CPC: investimento médio necessário para gerar um clique.
dados["cpc"] = divisao_segura(
    dados["investimento"], dados["cliques"]
)

# CPA: investimento médio necessário para gerar uma conversão.
dados["cpa"] = divisao_segura(
    dados["investimento"], dados["conversoes"]
)

# ROAS: receita gerada para cada unidade monetária investida em publicidade.
dados["roas"] = divisao_segura(
    dados["receita"], dados["investimento"]
)

# Seleciona algumas colunas para conferir os cálculos.
colunas_conferencia = [
    "data", "canal", "investimento", "impressoes", "cliques",
    "conversoes", "receita", "ctr_pct", "taxa_conversao_pct",
    "cpc", "cpa", "roas"
]

display(dados[colunas_conferencia].head().round(2))


## 7 Conferência manual de um registro

Vamos conferir o primeiro registro de Google Ads. Essa verificação ajuda a identificar erros nas fórmulas antes de analisar toda a base.


In [ ]:
# Seleciona a primeira linha da base tratada.
registro = dados.iloc[0]

# Calcula manualmente as métricas usando os valores do registro.
ctr_manual = registro["cliques"] / registro["impressoes"] * 100
taxa_manual = registro["conversoes"] / registro["cliques"] * 100
cpc_manual = registro["investimento"] / registro["cliques"]
cpa_manual = registro["investimento"] / registro["conversoes"]
roas_manual = registro["receita"] / registro["investimento"]

print(f"CTR: {ctr_manual:.2f}%")
print(f"Taxa de conversão: {taxa_manual:.2f}%")
print(f"CPC: R$ {cpc_manual:.2f}")
print(f"CPA: R$ {cpa_manual:.2f}")
print(f"ROAS: {roas_manual:.2f}")


## 8 Agrupamento correto por canal

Primeiro somaremos investimento, impressões, cliques, conversões e receita. Depois recalcularemos as taxas sobre os totais. Essa abordagem evita o erro de calcular a média simples de percentuais produzidos por volumes diferentes.


In [ ]:
# Lista das colunas que podem ser somadas por canal.
colunas_soma = [
    "investimento", "impressoes", "alcance", "cliques",
    "interacoes", "leads", "conversoes", "receita"
]

# Agrupa os registros por canal e soma as colunas selecionadas.
# min_count=1 mantém NaN quando todos os valores de um grupo forem ausentes.
por_canal = (
    dados.groupby("canal", as_index=False)[colunas_soma]
    .sum(min_count=1)
)

# Recalcula as métricas usando os totais de cada canal.
por_canal["ctr_pct"] = (
    por_canal["cliques"] / por_canal["impressoes"] * 100
)
por_canal["taxa_conversao_pct"] = (
    por_canal["conversoes"] / por_canal["cliques"] * 100
)
por_canal["cpc"] = (
    por_canal["investimento"] / por_canal["cliques"]
)
por_canal["cpa"] = (
    por_canal["investimento"] / por_canal["conversoes"]
)
por_canal["roas"] = (
    por_canal["receita"] / por_canal["investimento"]
)

# Ordena pelo volume de conversões, do maior para o menor.
por_canal = por_canal.sort_values(
    "conversoes", ascending=False
).reset_index(drop=True)

display(por_canal.round(2))


## 9 Identificação dos melhores resultados

Cada métrica responde a uma pergunta diferente. Para conversões, taxa de conversão e ROAS, valores maiores são melhores. Para CPA, valores menores representam menor custo por conversão.


In [ ]:
# idxmax retorna o índice da linha que possui o maior valor.
canal_mais_conversoes = por_canal.loc[
    por_canal["conversoes"].idxmax()
]

canal_melhor_taxa = por_canal.loc[
    por_canal["taxa_conversao_pct"].idxmax()
]

canal_maior_roas = por_canal.loc[
    por_canal["roas"].idxmax()
]

# idxmin retorna o índice da linha que possui o menor valor.
canal_menor_cpa = por_canal.loc[
    por_canal["cpa"].idxmin()
]

print(
    "Maior volume de conversões:",
    canal_mais_conversoes["canal"],
    "-", int(canal_mais_conversoes["conversoes"])
)
print(
    "Melhor taxa de conversão:",
    canal_melhor_taxa["canal"],
    f"- {canal_melhor_taxa['taxa_conversao_pct']:.2f}%"
)
print(
    "Maior ROAS:",
    canal_maior_roas["canal"],
    f"- {canal_maior_roas['roas']:.2f}"
)
print(
    "Menor CPA:",
    canal_menor_cpa["canal"],
    f"- R$ {canal_menor_cpa['cpa']:.2f}"
)


## 10 Visualização 1 Conversões por canal

O gráfico de barras facilita a comparação do volume acumulado de conversões.


In [ ]:
# Ordena os canais do maior para o menor volume de conversões.
ordem_conversoes = por_canal.sort_values(
    "conversoes", ascending=False
)

# Cria a figura e o eixo.
fig, ax = plt.subplots(figsize=(9, 5))

# Constrói o gráfico de barras.
sns.barplot(
    data=ordem_conversoes,
    x="canal",
    y="conversoes",
    color="#662C92",
    ax=ax
)

# Define título e nomes dos eixos.
ax.set_title("Conversões acumuladas por canal", fontsize=14, weight="bold")
ax.set_xlabel("Canal")
ax.set_ylabel("Quantidade de conversões")

# Escreve o valor acima de cada barra.
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", padding=3)

plt.tight_layout()
plt.show()


## 11 Visualização 2 Evolução semanal

O gráfico de linha permite verificar como as conversões mudaram ao longo do tempo em cada canal.


In [ ]:
# Agrupa por data e canal para garantir um valor por ponto do gráfico.
serie_temporal = (
    dados.groupby(["data", "canal"], as_index=False)["conversoes"]
    .sum()
)

fig, ax = plt.subplots(figsize=(10, 5))

sns.lineplot(
    data=serie_temporal,
    x="data",
    y="conversoes",
    hue="canal",
    marker="o",
    linewidth=2,
    ax=ax
)

ax.set_title("Evolução semanal das conversões", fontsize=14, weight="bold")
ax.set_xlabel("Data")
ax.set_ylabel("Conversões")
ax.legend(title="Canal", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


## 12 Visualização 3 Investimento e conversões

O gráfico de dispersão ajuda a investigar a associação entre investimento e conversões. O tamanho dos pontos representa a receita.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

sns.scatterplot(
    data=dados,
    x="investimento",
    y="conversoes",
    hue="canal",
    size="receita",
    sizes=(50, 260),
    alpha=0.8,
    ax=ax
)

ax.set_title("Investimento e conversões por registro", fontsize=14, weight="bold")
ax.set_xlabel("Investimento em publicidade (R$)")
ax.set_ylabel("Conversões")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()


## 13 Visualização 4 CPA e ROAS

CPA e ROAS devem ser lidos em conjunto. Um CPA baixo mostra eficiência na aquisição; um ROAS alto mostra maior receita em relação ao investimento em mídia.


In [ ]:
# Cria dois gráficos lado a lado.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.barplot(
    data=por_canal.sort_values("cpa"),
    x="canal", y="cpa",
    color="#EC0089",
    ax=axes[0]
)
axes[0].set_title("CPA por canal", weight="bold")
axes[0].set_xlabel("Canal")
axes[0].set_ylabel("CPA (R$)")
axes[0].tick_params(axis="x", rotation=20)

sns.barplot(
    data=por_canal.sort_values("roas", ascending=False),
    x="canal", y="roas",
    color="#662C92",
    ax=axes[1]
)
axes[1].set_title("ROAS por canal", weight="bold")
axes[1].set_xlabel("Canal")
axes[1].set_ylabel("ROAS")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


## 14 Correlação entre investimento e conversões

A correlação de Pearson varia de -1 a 1. Valores positivos indicam que as variáveis tendem a crescer juntas. Correlação não comprova causalidade.


In [ ]:
# Calcula a correlação linear entre investimento e conversões.
correlacao = dados["investimento"].corr(dados["conversoes"])

print(f"Correlação entre investimento e conversões: {correlacao:.3f}")

# Interpretação descritiva do valor absoluto da correlação.
valor_absoluto = abs(correlacao)

if valor_absoluto < 0.30:
    intensidade = "fraca"
elif valor_absoluto < 0.70:
    intensidade = "moderada"
else:
    intensidade = "forte"

direcao = "positiva" if correlacao > 0 else "negativa"

print(f"Associação linear {direcao} e {intensidade} nesta amostra.")
print("Atenção: correlação não demonstra que o investimento causou as conversões.")


## 15 Síntese automática dos resultados

Esta célula produz frases iniciais para apoiar a interpretação. A análise humana continua necessária para considerar objetivo da campanha, etapa do funil e limitações dos dados.


In [ ]:
# Localiza o canal de maior alcance.
maior_alcance = por_canal.loc[por_canal["alcance"].idxmax()]

# Localiza o canal de menor ROAS.
menor_roas = por_canal.loc[por_canal["roas"].idxmin()]

print("SÍNTESE DOS RESULTADOS")
print("-" * 60)
print(
    f"1. {canal_mais_conversoes['canal']} apresentou o maior volume, "
    f"com {int(canal_mais_conversoes['conversoes'])} conversões."
)
print(
    f"2. {canal_maior_roas['canal']} apresentou o maior ROAS: "
    f"{canal_maior_roas['roas']:.2f}."
)
print(
    f"3. {maior_alcance['canal']} apresentou o maior alcance: "
    f"{int(maior_alcance['alcance']):,} pessoas."
)
print(
    f"4. {menor_roas['canal']} apresentou o menor ROAS: "
    f"{menor_roas['roas']:.2f}; o resultado merece investigação adicional."
)


## 16 Interpretação orientada

Responda no relatório ou em novas células de texto:

1. Qual canal apresentou maior volume de conversões?
2. Qual canal apresentou melhor taxa de conversão?
3. Qual canal apresentou maior ROAS e menor CPA?
4. Que canal merece investigação adicional? Justifique com números.
5. A associação entre investimento e conversões comprova causalidade? Explique.
6. Escreva três insights usando a estrutura: **evidência → interpretação → relevância**.
7. Elabore duas recomendações testáveis.
8. Registre pelo menos três limitações da análise.

### Lembrete

- **Observação:** descreve o que aparece nos dados.
- **Hipótese:** propõe uma possível explicação.
- **Recomendação:** indica uma ação baseada nas evidências.


## 17 Exportação dos resultados

Salvaremos uma base tratada com as métricas e uma tabela consolidada por canal. No Colab, os arquivos serão disponibilizados para download.


In [ ]:
# Define os nomes dos arquivos de saída.
arquivo_dados_tratados = "campanhas_tratadas_com_metricas.csv"
arquivo_resumo = "metricas_por_canal.csv"

# Exporta sem incluir o índice numérico criado pelo Pandas.
dados.to_csv(
    arquivo_dados_tratados,
    index=False,
    encoding="utf-8-sig"
)

por_canal.to_csv(
    arquivo_resumo,
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos criados:")
print("-", arquivo_dados_tratados)
print("-", arquivo_resumo)

# Se estiver no Google Colab, inicia o download dos arquivos.
try:
    from google.colab import files

    files.download(arquivo_dados_tratados)
    files.download(arquivo_resumo)
except ImportError:
    print("Execução fora do Colab: os arquivos foram salvos na pasta atual.")


## 18 Encerramento

Antes de concluir, confirme:

- os dados foram carregados e tratados;
- as métricas foram calculadas corretamente;
- as taxas agregadas foram recalculadas sobre os totais;
- pelo menos três gráficos foram interpretados;
- os insights utilizam evidências numéricas;
- as recomendações são testáveis;
- as limitações foram registradas.

> **Cuidado conceitual:** ROAS não representa lucro. Ele compara receita atribuída e investimento em publicidade, sem considerar todos os custos do negócio.
